# OYO DataPortal Scraper Engine

This notebook runs the fully automated headless scraper pipeline powered by the new lead_v2.py architecture. **No sudo, no apt-get, no Xvfb required.** It uses Chrome's native `--headless=new` engine.

**Step 1:** Run the cell below to set up your environment (only needed once).

In [1]:
import subprocess
import sys

try:
    subprocess.run(["conda", "env", "create", "-f", "environment.yml"], check=True)
except subprocess.CalledProcessError:
    print('Env already exists or creation failed')

subprocess.run([sys.executable, "-m", "pip", "install", "playwright", "pandas", "requests", "beautifulsoup4", "lxml"], check=True)
subprocess.run([sys.executable, "-m", "playwright", "install", "chromium"], check=True)

Retrieving notices: done
Channels:
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done




==> WARNING: A newer version of conda exists. <==
    current version: 25.11.0
    latest version: 26.7.2

Please update conda by running

    $ conda update -n base -c conda-forge conda





python-3.10.21       | 24.4 MB   |            |   0% 
icu-78.3             | 23.0 MB   |            |   0% 

pandas-2.3.3         | 11.8 MB   |            |   0% 


numpy-base-2.2.5     | 7.4 MB    |            |   0% 



libopenblas-0.3.31   | 5.7 MB    |            |   0% 




openssl-3.5.8        | 5.5 MB    |            |   0% 





libstdcxx-15.2.0     | 3.7 MB    |            |   0% 






tk-8.6.15            | 3.4 MB    |            |   0% 







libgfortran5-15.2.0  | 1.5 MB    |            |   0% 








lxml-6.1.1           | 1.4 MB    |            |   0% 









sqlite-3.53.2        | 1.2 MB    |            |   0% 










setuptools-83.0.0    | 1.2 MB    |            |   0% 











pip-26.2.1           | 1.1 MB    |            |   0% 












ncurses-6.6          | 893 KB    |            |   0% 













xorg-libx11-1.8.13   | 817 KB    |            |   0% 














libgcc-15.2.0        | 803 KB    |            |   0% 















ld_impl_linux-6

CompletedProcess(args=['/opt/conda/bin/python3.13', '-m', 'playwright', 'install', 'chromium'], returncode=0)

**Step 2:** Inject your task directly into the pipeline queue.

In [7]:
import json, uuid, os
from datetime import datetime
import oyoms as om
 
# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
email = 'rajveer.singh@oyorooms.com'
excel_link = 'https://oyoenterprise-my.sharepoint.com/:x:/g/personal/rajveer_singh_oyorooms_com/IQDxU22aOU2xQqw27Ht4eRYhATvcug5Uyo_SkTVOG-eGKYg?e=m8lL87'
sheet_name = 'main'
 
os.makedirs('admin', exist_ok=True)
 
# ─────────────────────────────────────────────
# READ CONFIG VALUES FROM ONLINE SHEET
# Sheet layout (columns A / B):
#   locality         | "Texas"
#   search_terms     | ["Cafe"]
#   zip_codes        | ["78045", "77073"]
#   initial_wording  | "groupbookings_"
#   mode             | "emails"
# ─────────────────────────────────────────────
print("[*] Connecting to online config sheet...")
wb = om.WorkbookClient(email, url=excel_link)
 
# Read A1:B5 -> key/value pairs
raw = wb.get_range_data(sheet_name, range_address='A1:B5')
 
# ── DEBUG: show exactly what comes back so the parsing below can be
#    trusted. Safe to delete once things are confirmed working. ──
print(f"[DEBUG] type(raw) = {type(raw)}")
print(f"[DEBUG] raw =\n{raw!r}")
 
# Normalize into a dict {key: raw_value}, tolerant of the shape
# get_range_data actually returns (pandas DataFrame, list-of-lists,
# or list-of-dicts all behave differently when you iterate them).
config = {}
 
if hasattr(raw, "columns") and hasattr(raw, "iterrows"):
    # pandas DataFrame.
    cols = list(raw.columns)
    print(f"[DEBUG] DataFrame columns = {cols}")
    print(f"[DEBUG] DataFrame shape = {raw.shape}")
 
    # If get_range_data auto-promoted the first row to a header, that
    # row's data is sitting in raw.columns, not in the body — recover
    # it before reading the rest of the rows. But if pandas instead
    # defaulted to 'Unnamed: 0' / 'Unnamed: 1' style columns (i.e. no
    # header was promoted), cols[0]/cols[1] are NOT real data and must
    # be skipped, or you inject garbage keys like 'Unnamed: 0'.
    if len(cols) >= 2 and not str(cols[0]).startswith("Unnamed"):
        config[str(cols[0]).strip()] = cols[1]
 
    for _, r in raw.iterrows():
        vals = list(r.values)
        print(f"[DEBUG] row = {vals}")
        if len(vals) >= 2 and vals[0] is not None:
            config[str(vals[0]).strip()] = vals[1]
 
elif isinstance(raw, list):
    for row in raw:
        vals = list(row.values()) if isinstance(row, dict) else list(row)
        print(f"[DEBUG] row = {vals}")
        if len(vals) >= 2:
            config[str(vals[0]).strip()] = vals[1]
 
else:
    print("[!] Unrecognized format from get_range_data — check the [DEBUG] output above.")
 
print(f"[DEBUG] parsed config dict = {config}")
print(f"[DEBUG] config keys (repr) = {[repr(k) for k in config.keys()]}")
 
 
def parse_value(v):
    """
    Cells hold JSON-formatted text, e.g. '"Texas"' or '["Cafe"]'.
    Parse them into real Python types (str / list). Falls back to the
    raw string if a cell isn't valid JSON.
    """
    if v is None:
        return v
    if isinstance(v, (list, dict)):
        return v
    v = str(v).strip()
    try:
        return json.loads(v)
    except (json.JSONDecodeError, TypeError):
        return v
 
 
locality = parse_value(config.get('locality'))
search_terms = parse_value(config.get('search_terms'))
zip_codes = parse_value(config.get('zip_codes'))
# --- NEW DYNAMIC LOGIC ---
from final_queue_builder import STATE_ZIPS
if isinstance(zip_codes, list) and len(zip_codes) > 0:
    first_entry = str(zip_codes[0]).strip()
elif isinstance(zip_codes, str):
    first_entry = zip_codes.strip()
else:
    first_entry = ""

if first_entry and not first_entry.isdigit():
    if first_entry in STATE_ZIPS:
        print(f"[*] Detected state name '{first_entry}'. Loading zip codes from final_queue_builder...")
        zip_codes = STATE_ZIPS[first_entry]
    else:
        print(f"[!] '{first_entry}' is not numeric and not found in STATE_ZIPS. Proceeding with caution.")
# -------------------------
initial_wording = parse_value(config.get('initial_wording'))
mode = parse_value(config.get('mode'))
 
print(f"[*] locality         = {locality}")
print(f"[*] search_terms     = {search_terms}")
print(f"[*] zip_codes        = {zip_codes}")
print(f"[*] initial_wording  = {initial_wording}")
print(f"[*] mode             = {mode}")
 
# Fail loudly instead of silently writing a task full of Nones.
missing = [name for name, val in [
    ("locality", locality),
    ("search_terms", search_terms),
    ("zip_codes", zip_codes),
    ("initial_wording", initial_wording),
    ("mode", mode),
] if val is None]
 
if missing:
    raise RuntimeError(
        f"[!] Missing config value(s): {missing}. "
        f"Check the [DEBUG] output above to see what came back from "
        f"get_range_data and why these keys weren't found in config."
    )
 
# ─────────────────────────────────────────────
# BUILD TASK
# ─────────────────────────────────────────────
tasks = []
tasks.append({
    "id": uuid.uuid4().hex[:8],
    "created_at": datetime.now().isoformat(),
    "initial_wording": initial_wording,
    "search_terms": search_terms,
    "locality": locality,
    "zip_codes": zip_codes,
    "mode": mode,
    "completed_zips": {},
    "status": "pending"
})
 
with open('admin/tasks.json', 'w') as f:
    json.dump(tasks, f, indent=4)
 
print("\n[+] Setup complete! Task injected.")

[*] Connecting to online config sheet...
[DEBUG] type(raw) = <class 'pandas.DataFrame'>
[DEBUG] raw =
          locality               Texas
0     search_terms      ["car washer"]
1        zip_codes  ["78045", "77073"]
2  initial_wording      groupbookings_
3             mode              emails
[DEBUG] DataFrame columns = ['locality', 'Texas']
[DEBUG] DataFrame shape = (4, 2)
[DEBUG] row = ['search_terms', '["car washer"]']
[DEBUG] row = ['zip_codes', '["78045", "77073"]']
[DEBUG] row = ['initial_wording', 'groupbookings_']
[DEBUG] row = ['mode', 'emails']
[DEBUG] parsed config dict = {'locality': 'Texas', 'search_terms': '["car washer"]', 'zip_codes': '["78045", "77073"]', 'initial_wording': 'groupbookings_', 'mode': 'emails'}
[DEBUG] config keys (repr) = ["'locality'", "'search_terms'", "'zip_codes'", "'initial_wording'", "'mode'"]
[*] locality         = Texas
[*] search_terms     = ['car washer']
[*] zip_codes        = ['78045', '77073']
[*] initial_wording  = groupbookings_
[*] mo

**Step 3:** Start the Pipeline Engine!

In [5]:
pip install ./oyoms-0.1.8.5-py3-none-any.whl

Processing ./oyoms-0.1.8.5-py3-none-any.whl
  Using cached msal-1.38.0-py3-none-any.whl.metadata (11 kB)
  Using cached azure_identity-1.25.3-py3-none-any.whl.metadata (91 kB)
  Using cached azure_storage_blob-12.30.1-py3-none-any.whl.metadata (26 kB)
  Using cached msgraph_core-0.2.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached appdirs-1.4.4-py2.py3-none-any.whl.metadata (9.0 kB)
  Using cached pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached azure_core-1.41.0-py3-none-any.whl.metadata (49 kB)
  Using cached msal_extensions-1.3.1-py3-none-any.whl.metadata (7.8 kB)
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached msgraph_core-0.2.2-py3-none-any.whl (15 kB)
Using cached appdirs-1.4.4-py2.py3-none-any.whl (9.6 kB)
Using cached azure_identity-1.25.3-py3-none-any.whl 

In [8]:
import os
import subprocess
import sys

os.makedirs('golden_profile', exist_ok=True)
subprocess.run([sys.executable, "server_pipeline.py", "--golden-dir", "./golden_profile"])


   SERVER PIPELINE — Fully Automated Scrape Run
  Profiles  : 5
  Golden dir: ./golden_profile
  OS        : Linux

[Phase 0] Loading task configuration...
  [OK] Loaded task: e3a4278c | Texas | 2 zips

[Phase 1] Validating golden profile at: ./golden_profile
  [OK] Golden profile found.

[Phase 2] Virtual display skipped — Using Chrome's --headless=new mode instead.

[Phase 3] Cloning 5 worker profiles from golden profile...
  [OK] Cloned profile_1
  [OK] Cloned profile_2
  [OK] Cloned profile_3
  [OK] Cloned profile_4
  [OK] Cloned profile_5
  [OK] 5 worker profiles ready.

[Phase 4] Google Maps Scraping
  Terms: 1 | Zips: 2 | Workers: 5
  Queuing 2 tasks (0 already done).

[Worker 2] Starting.
[DEBUG] Found service worker URL: chrome-extension://iandhhofjjefcddffjfabpagigfigafc/src/background.js
[Worker 2] -> car washer in 78045 Texas
[Worker 2]    State: list
[Worker 2] Error on 'car washer in 78045 Texas': Locator.click: Timeout 30000ms exceeded.
Call log:
  - waiting for get_by_

CompletedProcess(args=['/opt/conda/bin/python3.13', 'server_pipeline.py', '--golden-dir', './golden_profile'], returncode=0)

In [9]:
# The % prefix tells Jupyter to run this as a package installation command
%pip install conda-pack

import conda_pack

# This creates the .tar.gz archive while ignoring the broken pip files
conda_pack.pack(output="my_environment.tar.gz", ignore_missing_files=True, force=True)

print("Finished packaging successfully!")

Note: you may need to restart the kernel to use updated packages.


/opt/conda/lib/python3.13/site-packages/conda_pack/core.py:58: UserWarning: 
Conda-managed packages were found without entries in the package cache. This
is usually due to `conda clean -p` being unaware of symlinked or copied
packages. Uncached packages:

- lzo='2.10'   https://conda.anaconda.org/conda-forge/linux-64/lzo-2.10-h280c20c_1002.conda
- zstd='1.5.7'   https://conda.anaconda.org/conda-forge/linux-64/zstd-1.5.7-hb78ec9c_6.conda
- archspec='0.2.5'   https://conda.anaconda.org/conda-forge/noarch/archspec-0.2.5-pyhd8ed1ab_0.conda
- mamba='2.4.0'   https://conda.anaconda.org/conda-forge/linux-64/mamba-2.4.0-h7ae174a_1.conda
- libstdcxx-ng='15.2.0'   https://conda.anaconda.org/conda-forge/linux-64/libstdcxx-ng-15.2.0-hdf11a46_16.conda
- libmambapy='2.4.0'   https://conda.anaconda.org/conda-forge/linux-64/libmambapy-2.4.0-py313hb20a507_1.conda
- jsonpointer='3.0.0'   https://conda.anaconda.org/conda-forge/noarch/jsonpointer-3.0.0-pyhcf101f3_3.conda
- frozendict='2.4.7'   https://con

Finished packaging successfully!
